En este apartado se van a procesar las reglas de cada uno de los apartados correspondientes.

Utilidades: normalizar respuestas y aplicar reglas avisando si algo no coincide.

In [ ]:
import re
import unicodedata
import pandas as pd
from pathlib import Path

# Busca la carpeta "data" subiendo desde donde se ejecuta el notebook
# (funciona igual desde la raíz del proyecto, rules/ o distributions/)
DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())

RUTA_LIMPIA = DATA / "FORMULARIO DE CONTEXTO limpio.csv"
RUTA_CONDICIONES = DATA / "CONDICIONES MATERIALES.csv"

def normalizar(texto):
    """Minúsculas, sin acentos, sin espacios dobles y sin puntuación al final.
    Así 'Muy Difícil', 'Muy dificil' y 'Muy difícil.' cuentan como la misma respuesta."""
    if pd.isna(texto):
        return texto
    texto = unicodedata.normalize("NFKD", str(texto).replace("\u202f", " "))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"\s+", " ", texto).strip().lower()
    return texto.rstrip(".,;: ")

def aplicar_regla(df, columna, regla):
    """Convierte una columna usando un diccionario de regla.
    Si alguna respuesta no está en el diccionario, lo avisa en lugar de dejarla en blanco sin decir nada."""
    regla_norm = {normalizar(k): v for k, v in regla.items()}
    original = df[columna]
    convertida = original.map(lambda v: regla_norm.get(normalizar(v), float("nan")))
    sin_regla = original.notna() & convertida.isna()
    if sin_regla.any():
        print(f"AVISO '{columna}': respuestas sin regla -> {sorted(original[sin_regla].astype(str).unique())}")
    df[columna] = convertida.astype(float)

Reglas para la encuesta en el apartado de Capital 

In [ ]:
# REGLAS PARTICULARES DEL HOGAR
df = pd.read_csv(RUTA_LIMPIA, encoding="utf-8-sig")

regla_tamano_de_la_vivienda = {
    "Menos de 50 m²: Corresponde a departamentos pequeños o viviendas de interés social" : 0.0,
    "Entre 50 y 80 m²: Es el estándar de muchas casas de fraccionamientos o viviendas urbanas promedio" : 0.25,
    "Entre 81 y 120 m²: Viviendas de tamaño medio con mayor número de habitaciones" : 0.50,
    "Entre 121 y 200 m²: Viviendas amplias, usualmente de dos niveles o con terrenos extensos" : 0.75,
    "Más de 200 m²: Residencias de grandes dimensiones o propiedades con terrenos amplios." : 1.0
}

regla_personas_en_casa = {
    "1" : 0,
    "De 2 a 3" : 0.25,
    "De 3 a 4" : 0.50,
    "De 4 a 5" : 0.75,
    "Mas de 5" : 1
}

regla_de_la_habitacion = {
    "Solo yo" : 1,
    "1" : 1,          # 3 alumnos respondieron "1" (= solo ellos en la habitación)
    "Dos personas" : 0.75,
    "Entre dos y tres personas" : 0.50,
    "Entre tres y cuatro personas" : 0.25,
    "Cinco o mas" : 0
}

regla_de_la_liquidez = {
    'Muy facil' : 0.0,
    'Facil' : 0.25,
    'Moderado' : 0.50,
    'Dificil' : 0.75,
    'Muy dificil' : 1.0
}

regla_promedio_ingresos_mensuales = {
    'Menos de $7,500 MXN' : 1.0,
    'De $7,501 a $15,000 MXN' : 0.75,
    'De $15,001 a $22,500 MXN' : 0.50,
    'De $22,501 a $37,500 MXN' : 0.25,
    'Mas de $37,500 MXN' : 0.0
}

regla_del_tiempo_de_transporte = {
    'Menos de 30 minutos' : 0.0,
    'Entre 30 minutos y 1 hora' : 0.25,
    'Entre 1 hora y 1 hora 30 minutos' : 0.50,
    'Entre 1 hora 30 minutos y 2 horas' : 0.75,
    'Mas de 2 horas diarias' : 1.0
}

regla_costos_transporte_por_semana = {
    'Menos de $150 MXN a la semana.' : 0.0,
    'Entre $250 y $400 MXN a la semana.' : 0.25,
    'Entre $450 y $550 MXN a la semana.' : 0.50,
    'Entre de $600 a $750 MXN a la semana.' : 0.75,
    'Mas de $750 MXN a la semana' : 1.0
}

# Aplicamos aqui la conversion a numeros
aplicar_regla(df, "Tamano casa", regla_tamano_de_la_vivienda)
aplicar_regla(df, "Personas hogar", regla_personas_en_casa)
aplicar_regla(df, "Personas habitacion", regla_de_la_habitacion)
aplicar_regla(df, "Dificultad liquidez", regla_de_la_liquidez)
aplicar_regla(df, "Ingreso mensual", regla_promedio_ingresos_mensuales)
aplicar_regla(df, "Tiempo transporte", regla_del_tiempo_de_transporte)
aplicar_regla(df, "Gasto transporte", regla_costos_transporte_por_semana)

#----------------
# Paso A: Vamos a calcular el "Espacio disponible"
Espacio = (df["Tamano casa"] + df["Personas habitacion"]) / 2

# Paso B: Calculamos el Hacinamiento
Hacinamiento = df["Personas hogar"]

# Paso C: Formula Final.
df["Indice_Confort_Hogar"] = ((Espacio - Hacinamiento + 1) / 2).round(2)

print(df[["Tamano casa", "Personas hogar", "Personas habitacion", "Indice_Confort_Hogar"]].head(10))

columnas_a_guardar = [
    "Nombre", "Agua", "Luz",
    "Internet", "Equipo propio", "Equipo lento",
    "Banos", "Tamano casa", "Personas hogar",
    "Personas habitacion", "Seguridad transporte", "Gasto transporte",
    "Tiempo transporte", "Ingreso mensual", "Dificultad liquidez",
    "Indice_Confort_Hogar"
]

condiciones = df[columnas_a_guardar]
vacios = condiciones.isna().sum()
print("\nVacíos por columna:", "ninguno" if vacios.sum() == 0 else vacios[vacios > 0].to_dict())

condiciones.to_csv(RUTA_CONDICIONES, index=False, encoding="utf-8-sig")
print("¡Archivo CONDICIONES MATERIALES guardado!")

Reglas para la encuesta del Capital de Salud

In [ ]:
# REGLAS DEL CAPITAL DE SALUD
df = pd.read_csv(RUTA_LIMPIA, encoding="utf-8-sig")

regla_gastos_mensuales_salud = {
    'No requiero' : 0.0,
    'Gasto mínimo' : 0.33,
    'Gasto moderado' : 0.67,
    'Gasto elevado' : 1.0
}

regla_comodidas_completas_por_dia = {
    'Menos de una' : 1.0,
    'Dos o tres' : 0.50,
    'Mas de tres' : 0.0
}

regla_inicio_clases_sin_comer = {
    'Frecuentemente (al menos 4 o 5 veces de la semana)' : 1.0,
    'Ocasionalmente (al menos 3 veces por semana)' : 0.75,
    'Rara vez (al menos 1 vez por semana)' : 0.50,
    'Casi nunca (muy rara vez por mes)' : 0.25,
    'Nunca' : 0.0
}

regla_consumo_alcohol_tabaco = {
    'Nunca: No consumo ninguna de estas sustancias bajo ninguna circunstancia.' : 0.0,
    'Ocasionalmente: Solo en eventos sociales (menos de una vez al mes).' : 0.25,
    'Rara vez: Una o dos veces al mes, de forma esporádica y sin seguir una rutina.' : 0.50,
    'Regularmente: De forma semanal, generalmente los fines de semana o días de descanso.' : 0.75,
    'Frecuentemente: Consumo diario o casi todos los días de la semana.' : 1.0
}

regla_presupuesto_consumo = {
    'Nada (0%): No asigno recursos económicos a estos productos' : 0.0,
    'Mínimo (Menos del 5%): Representa un gasto marginal, como un café ocasional' : 0.25,
    'Moderado (Entre el 5% y el 15%): Es un gasto recurrente que ya está contemplado' : 0.50,
    'Considerable (Entre el 16% y el 30%): Una parte importante del presupuesto se destina a esto' : 0.75,
    'Significativo (Más del 30%): El consumo de estas sustancias es una de mis prioridades' : 1.0
}

regla_frecuencia_consumo_ciclo_sueno = {
    'Nunca: Mi consumo no tiene ningún impacto negativo en mi descanso' : 0.0,
    'Rara vez: Ha sucedido en eventos muy aislados' : 0.25,
    'Ocasionalmente: Sucede algunas veces por mes' : 0.50,
    'Recuentemente: Es un efecto recurrente que altera mi horario de sueño varias veces al mes' : 0.75,
    'Muy frecuentemente: El impacto es constante o diario' : 1.0
}

regla_dependencia_energizantes = {
    'Nula: Cumplo con mis entregas y estudio de manera efectiva' : 0.0,
    'Baja: Solo los utilizo en situaciones excepcionales de carga extrema' : 0.25,
    'Moderada: Son un apoyo recurrente en mi rutina' : 0.50,
    'Alta: Los consumo de forma habitual' : 0.75,
    'Total: Siento que no podría finalizar mis deberes si no consumo estas sustancias.' : 1.0
}

aplicar_regla(df, "Gastos salud", regla_gastos_mensuales_salud)
aplicar_regla(df, "Comidas diarias", regla_comodidas_completas_por_dia)
aplicar_regla(df, "Falta desayuno", regla_inicio_clases_sin_comer)
aplicar_regla(df, "Consumo sustancias", regla_consumo_alcohol_tabaco)
aplicar_regla(df, "Gasto sustancias", regla_presupuesto_consumo)
aplicar_regla(df, "Impacto sustancias", regla_frecuencia_consumo_ciclo_sueno)
aplicar_regla(df, "Dependencia estimulantes", regla_dependencia_energizantes)

columnas_salud = ["Gastos salud", "Comidas diarias", "Falta desayuno", "Consumo sustancias",
                  "Gasto sustancias", "Impacto sustancias", "Dependencia estimulantes"]
vacios = df[columnas_salud].isna().sum()
print("Vacíos por columna:", "ninguno" if vacios.sum() == 0 else vacios[vacios > 0].to_dict())
print(df[columnas_salud].head())